# Cell Dataset Creation

Will download and prepare the cell dataset for training

## Download and extract dataset

In [ ]:
import requests
import pathlib
import albumentations as alt
import zipfile

data_path = pathlib.Path('../data/')
data_path.mkdir(exist_ok=True)
zip_path = data_path / "pbc_dataset.zip"
extract_path = data_path / "pbc_dataset"
extract_path.mkdir(exist_ok=True)
pbc_base_path = extract_path / "dataset"

In [ ]:
URLS = {
    'pbc_dataset.zip': 'https://zenodo.org/records/17333317/files/dataset.zip?download=1',
    'pbc_meta.csv': "https://zenodo.org/records/17333317/files/metadata.csv?download=1",
}
for filename, url in URLS.items():
    response = requests.get(url)
    if response.status_code == 200:
        with open(data_path / filename, 'wb') as f:
            f.write(response.content)
    else:
        print(response)

In [ ]:

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
import pandas as pd

df = pd.read_csv(data_path / 'pbc_meta.csv')
df["split"] = df['path'].apply(lambda x: x.split('/')[0])
df["label"] = df['path'].apply(lambda x: x.split('/')[-1])
df['path'] = df['path'] + '/' + df['image_name']
df = df.drop(columns=['patient_id', 'image_name'])

df.head()

In [ ]:
sorted_labels = sorted(df['label'].unique().tolist())
label_id_map = {l:i for i, l in enumerate(sorted_labels)}
df['label'] = df['label'].apply(lambda x: label_id_map[x])
df.head()